In [0]:
%pip install spark

In [0]:
df = spark.read.option("header", True).option("inferSchema", True).csv("/Volumes/workspace/default/home_credit/application_train.csv")

In [0]:
display(df)

In [0]:
display(df.describe())

In [0]:
print(f"El DataFrame tiene {len(df.columns)} columnas.")


In [0]:
df.groupBy('CNT_CHILDREN').count().orderBy('CNT_CHILDREN').show() 

In [0]:
df.filter(df.CNT_CHILDREN >= 14).select('CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'AMT_INCOME_TOTAL').show()

Después de una revisión a la variable CNT_CHILDREN, se decidió mantener los valores que se consideran outliers dentro del dataset de entrenamiento, puesto que se va a usar el modelo LightGBM y para este modelo no representa importancia valores extremos, como en este caso familias con más de 14 hijos o 20 miembros de la familia, ya que se encarga de clasificar y esto no modifica su funcionamiento.

In [0]:
df.groupBy("TARGET").count().show()

In [0]:
df.createOrReplaceTempView("df")

In [0]:
%sql

SELECT
    TARGET, 
    count(TARGET),
    (COUNT(TARGET) * 100) / SUM(count(TARGET)) OVER() as porcentaje_total
FROM df
GROUP BY TARGET

In [0]:
pos = 24825
neg = 282686

scale = neg/pos
print(f"La cantidad de clientes que no pagan es {neg} y la cantidad de clientes que pagan es {pos}")
print(f"El factor de escala es {scale}")

La cantidad de clientes que no pagan es 282686 y la cantidad de clientes que pagan es 24825
El factor de escala es 11.387150050352467

In [0]:
df_boxplo = df.select('TARGET', "EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3").toPandas()

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt

pdf_long = df_boxplo.melt(id_vars='TARGET', value_vars=['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3'],
                     var_name='variable', value_name='puntaje')

plt.figure(figsize=(10, 6))
sns.boxplot(data=pdf_long, x='variable', y='puntaje', hue='TARGET')
plt.xticks([0, 1, 2], ['Score 1', 'Score 2', 'Score 3'])
plt.ylabel('Puntaje')
plt.title('Score externo 1/2/3 por TARGET')
plt.show()

In [0]:
df_boxplo.groupby('TARGET')[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].describe().transpose()

**NOTA:** Se muestra como las personas que si pagan tienen valñores de puntuaje externos mucho mása altos, lo que es esperable, ya ue tiene coherencia, además cuetnan con medianas más altas en los diferents puntajes externos.

**NOTA:** Los outlier de EXT_SOURCE_2 de las personas que no pagan, ya que son personas que a pesar de tener un puntuaje bastante alto, lo que signficaría que son confiables, igualmente dieron por default, así que EXT_SOURCE es una beuna variable apra el atregt, pero se encesita apoyo de otras más. 

**NOTA:** En el describe se idnetifica que para el EXT_SCORE_1 hay un total de 124,079 + 10,054 = 134,133, dando a entender que de los 307,000 regsitros que hay, casi la mitad pueden no contar con puntaje crediticio externo

In [0]:
import seaborn as sns
import matplotlib.pyplot as plt
pdf2 = df.select('TARGET', 'AMT_INCOME_TOTAL').toPandas()

plt.figure(figsize=(8, 6))
sns.boxplot(data=pdf2, x='TARGET', y='AMT_INCOME_TOTAL')
plt.xticks([0, 1], ['No default', 'DEFAULT'])
plt.ylabel('Ingreso anual')
plt.title('Ingreso anual por TARGET')
plt.ylim([0, 350000])
plt.show()

In [0]:
pdf2.groupby('TARGET')['AMT_INCOME_TOTAL'].describe().transpose()

**NOTA:** No hay mucha difrencia en las medias, las cajas se solapan mucho en el mismo lugar, así que sola no será un variable muy relevante, pero será importante para el **debt-to-income ratio**

In [0]:
df.select('DAYS_EMPLOYED').summary().show()

In [0]:
df.filter(df.DAYS_EMPLOYED == 365243).count()

La variable `DAYS_EMPLOYED` cuenta con valores negativos, ya que representa los días trabajados por el cliente hasta el día de solicitar el crédito. Hay registros con un mismo valor positivo, que es 365243, que es una manera de decir que esa persona no cuenta con días trabajados hasta la solicitud del crédito.

In [0]:
from pyspark.sql import functions as F

df.groupBy('NAME_HOUSING_TYPE').agg(F.avg('TARGET').alias('default_rate'),F.count('TARGET').alias('count')).orderBy('default_rate', ascending=False).show()

In [0]:
df.groupBy('NAME_FAMILY_STATUS').agg(
    F.avg('TARGET').alias('default_rate'),
    F.count('TARGET').alias('count')
).orderBy('default_rate', ascending=False).show()

### Análisis: NAME_HOUSING_TYPE y NAME_FAMILY_STATUS vs TARGET

**NAME_HOUSING_TYPE**

Encontramos que hay un mayor riesgo de default en ciertos grupos, por ejemplo las personas que rentan departamentos y las que viven con sus padres tienen un 12%. De ahí para abajo la cifra disminuye, pero esos son los casos más relevantes.

Si bien es cierto que la mayor cantidad de personas son las que tienen un departamento o una casa (casi 272.000 personas del dataset completo), ellos solo representan un 7%, por lo que no es tan relevante. Hay que tener esto en cuenta porque entre el máximo y el mínimo hay una diferencia de 5.7 puntos porcentuales.

**NAME_FAMILY_STATUS**

La hipótesis era que las parejas casadas estarían en un punto más seguro, al tener una mayor cantidad de ingresos conjuntos existe un mayor potencial de no caer en default por la capacidad económica.

Sin embargo en el gráfico nos encontramos con lo siguiente:

1. Civil Marriage (unión libre): tiene el porcentaje más alto, aunque no llega al 10% (se sitúa en un 9,9%), sí representa un impacto considerable.
2. Parejas casadas: tal como se preveía, muestran un porcentaje bajo del 7%.
3. Viudas: tienen un 5%.

En general los resultados son bastante variados y por encima de estas categorías ninguna supera el ratio del 10%. Cabe notar que existen dos registros que no cuentan con información en NAME_FAMILY_STATUS, por lo que no se pudieron incluir en este análisis.

In [0]:
df.select('AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_ANNUITY', 'AMT_INCOME_TOTAL').toPandas().corr()

In [0]:
df.select('AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_ANNUITY', 'AMT_INCOME_TOTAL').toPandas().isnull().sum()

In [0]:
cols = ['AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_ANNUITY', 'TARGET']
pdf = df.select(cols).toPandas()

import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 1, figsize=(10, 21))
for i, col in enumerate(['AMT_CREDIT', 'AMT_GOODS_PRICE', 'AMT_ANNUITY']):
    sns.boxplot(data=pdf, x='TARGET', y=col, ax=axes[i])
    axes[i].set_title(col)

axes[0].set_ylim(0, 1650000)
axes[1].set_ylim(0, 1500000)
axes[2].set_ylim(0, 80000)
plt.tight_layout()
plt.show()

### Análisis: AMT_CREDIT, AMT_GOODS_PRICE, AMT_ANNUITY vs TARGET

Hablando de las tres variables de amount, nos encontramos con que la variable AMT_GOODS_PRICE (precio del bien) y la variable AMT_CREDIT (el monto del crédito) tienen una relación muy estrecha, con un 98% de correlación.

Al ser una correlación tan grande, se ha decidido generar variables nuevas para el feature engineering:

1. Downpayment ratio: usando AMT_GOODS_PRICE y AMT_CREDIT
2. Payment to income: usando AMT_ANNUITY y AMT_INCOME_TOTAL

Por otro lado, encontramos que estas variables casi no tienen valores nulos. AMT_ANNUITY tiene 12 registros nulos, que se eliminarán en el feature engineering. AMT_GOODS_PRICE tiene 278 registros nulos, que esperamos imputar con el promedio, calculado únicamente sobre el set de train antes de aplicarlo a train y validación.

Por otro lado, si vemos las gráficas de los boxplots de las tres variables, nos damos cuenta de lo siguiente:

1. En la variable target, ya sea por default o no default, las cajas están muy compactas.
2. Aunque hay demasiados outliers, las cajas como tal se mantienen compactas.
3. Su mediana, sus tamaños y su rango se parecen demasiado.

En conclusión, estas variables no son tan fuertes por sí solas, pero sirven mucho para obtener ratios. Estos son valores que ninguna otra variable mide actualmente y resultan bastante relevantes.

## Conclusiones generales — EDA P2 (Home Credit Default Risk)

Como conclusión general del EDA, encontramos que el dataset es bastante robusto. Contamos con una buena cantidad de variables (121 features, 122 columnas incluyendo el target). Prácticamente no tenemos valores nulos en la mayoría de columnas, pero el dataset está bastante desbalanceado:

1. Registros totales: 307.511 registros.
2. Desbalanceo de clases:
   - Clase 0 (no default): 282.686 registros, 91.93%.
   - Clase 1 (default), que es la que necesitamos identificar: 24.825 registros, 8.07%.

Sobre las variables específicas, identificamos lo siguiente:

**External Sources**

Encontramos que estos puntajes externos de otros bancos son bastante relevantes y tienen un poder de predicción fuerte. Al agrupar los datos, las medianas están muy separadas y las cajas no se solapan, con pocos outliers (quizás algunos en el Score 2 que debemos revisar bien).

Una conclusión interesante es que existen outliers en el External Source 2 de personas que no pagan; a pesar de tener un puntaje alto, dieron default, lo que indica que no son tan confiables por sí solos. Es una buena variable para el target, pero necesita el apoyo de otras.

**External Source 1**

Tiene 134.133 registros con dato (124.079 en clase 0 + 10.054 en clase 1) sobre 307.511 totales. Esto significa que 173.378 registros (56.4%) no cuentan con este puntaje crediticio externo, es decir, no tienen historial pasado o están limpios. Es el missing más alto de los tres External Source y se evaluará flag de "sin score" + estrategia de imputación en feature engineering.

**AMT Income Total**

No encontramos una gran variedad en su mediana y las cajas son casi del mismo tamaño. Aunque las colas superiores presentan bastantes outliers, la diferencia entre el primer y tercer cuartil de ambas categorías (0 y 1) es prácticamente mínima. Sin embargo, esta variable puede ser importante para calcular el payment-to-income ratio, algo que manejaremos en el feature engineering del siguiente notebook.

**DAYS_EMPLOYED**

Encontramos un valor importante a tener en cuenta: normalmente todos son negativos, pues representan los días trabajados hasta el momento de pedir el crédito. Sin embargo, hay registros con el valor 365.243 en positivo; entendemos que esto corresponde a personas que nunca han trabajado al momento de la solicitud. Manejaremos este punto en el feature engineering del siguiente notebook (reemplazo por null + columna FLAG_EMPLOYED).

**NAME_HOUSING_TYPE**

El ratio de default es bastante alto en las personas que viven en renta o con sus padres, con 12% y 11% respectivamente. Aunque no son cantidades tan grandes, sí representan una mayor probabilidad de impago frente al resto de categorías.

**NAME_FAMILY_STATUS**

Las personas en unión libre (civil marriage) tienen el ratio de default más alto, cercano al 10%. Las personas casadas son las que más se registran en el dataset (196.432 personas) y presentan un 7% de probabilidad de default. Dos registros sin información (Unknown) se descartarán en feature engineering.

**Variables de monto (AMT_CREDIT, AMT_GOODS_PRICE, AMT_ANNUITY, AMT_INCOME_TOTAL)**

Encontramos una correlación del 98% entre AMT_GOODS_PRICE y AMT_CREDIT, y de 77% entre AMT_CREDIT y AMT_ANNUITY. Por ello, se decidió crear dos variables nuevas para el feature engineering:

1. **downpayment_ratio** (AMT_GOODS_PRICE / AMT_CREDIT) — captura cuánta cuota inicial puso el cliente, información que ninguna de las dos variables originales aporta por separado. Se eliminarán AMT_CREDIT y AMT_GOODS_PRICE del set de features tras crear esta ratio.
2. **payment_to_income** (AMT_ANNUITY / AMT_INCOME_TOTAL) — captura la carga relativa de la deuda frente a la capacidad de pago. AMT_ANNUITY y AMT_INCOME_TOTAL se mantienen también como variables independientes, dado que su correlación (77% y por debajo) no llega al nivel de redundancia de AMT_CREDIT/AMT_GOODS_PRICE.

También identificamos valores nulos a manejar en feature engineering:
- 12 registros nulos en AMT_ANNUITY — se eliminarán (volumen insignificante).
- 278 registros nulos en AMT_GOODS_PRICE — se imputarán con la media, calculada únicamente sobre el set de train y aplicada después a train y validación.

Finalmente, al analizar AMT_INCOME_TOTAL, AMT_CREDIT, AMT_GOODS_PRICE y AMT_ANNUITY divididos por TARGET, notamos que las cajas en el boxplot están muy solapadas y sus medianas son casi idénticas. Si bien el tamaño de las cajas varía ligeramente dependiendo de si hay default o no, y todas presentan bastantes outliers, no parecen ser tan relevantes por sí solas. Sin embargo, nos servirán para generar los ratios mencionados arriba, que sí capturan señal relevante para el modelo.

**Jerarquía de poder predictivo identificada:**
1. EXT_SOURCE_1/2/3 — predictor fuerte
2. NAME_HOUSING_TYPE, NAME_FAMILY_STATUS — señal moderada
3. AMT_INCOME_TOTAL, AMT_CREDIT, AMT_GOODS_PRICE, AMT_ANNUITY — señal débil individualmente, valiosas en ratios